# SpliceAI

![SpliceAI](https://proto-bio.github.io/proto-assets/images/tool/spliceai/hero.png)

SpliceAI is a deep neural network that predicts splicing from primary DNA sequence. It serves two purposes: scoring genetic variants for their splice-altering effect via delta scores (probabilities that the variant gains or loses an acceptor/donor site), and predicting raw per-position acceptor/donor splice-site probabilities directly from a DNA sequence. It is widely used to interpret the splicing impact of non-coding and synonymous variants in clinical and research genomics.

- Paper: [Jaganathan et al., Cell 2019](https://doi.org/10.1016/j.cell.2018.12.015)
- Repository: [Illumina/SpliceAI](https://github.com/Illumina/SpliceAI)

In [1]:
from proto_tools.tools import (
    run_spliceai_score,
    run_spliceai_predict,
    SpliceAIScoreInput,
    SpliceAIScoreConfig,
    SpliceAIVariant,
    SpliceAIPredictInput,
    SpliceAIPredictConfig,
)
from proto_tools.utils.notebook_docs import display_api_reference

In [2]:
# API reference for the variant-scoring tool
display_api_reference("spliceai-score", "input", "run_spliceai_score")
display_api_reference("spliceai-score", "config", "run_spliceai_score")
display_api_reference("spliceai-score", "output", "run_spliceai_score")

**Input** — `SpliceAIScoreInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `variants` | `list[proto_tools.tools.rna_splicing.spliceai.spliceai_score.SpliceAIVariant]` | required | Variants to score for splice-altering effects |

**Config** — `SpliceAIScoreConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run inference on |
| `timeout` | `int \| None` | `600` | Maximum execution time in seconds. None waits indefinitely. |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `reference_fasta` | `str \| None` | `None` | Path (or AssetRef) to the reference genome FASTA; required at call time |
| `annotation` | `str` | `'grch38'` | 'grch37'/'grch38' (bundled) or path to a custom gene annotation file |
| `max_distance` | `int` | `50` | Max distance (bp) between variant and gained/lost splice site (the -D flag) |
| `mask` | `bool` | `False` | Mask annotated-gain / unannotated-loss scores (the -M flag) |

**Output** — `SpliceAIScoreOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `list[proto_tools.tools.rna_splicing.spliceai.spliceai_score.SpliceAIVariantResult]` | required | Per-variant SpliceAI scores (1:1 with input variants) |

In [3]:
# API reference for the raw splice-site prediction tool
display_api_reference("spliceai-predict", "input", "run_spliceai_predict")
display_api_reference("spliceai-predict", "config", "run_spliceai_predict")
display_api_reference("spliceai-predict", "output", "run_spliceai_predict")

**Input** — `SpliceAIPredictInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | DNA sequence(s) to predict splice-site probabilities for |

**Config** — `SpliceAIPredictConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run inference on |
| `timeout` | `int \| None` | `600` | Maximum execution time in seconds. None waits indefinitely. |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |

**Output** — `SpliceAIPredictOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `predictions` | `list[list[list[float]]]` | required | Per-sequence, per-position [neither, acceptor, donor] probabilities |

## Variant scoring

Variant scoring needs a reference genome FASTA and a gene annotation: SpliceAI extracts the wild-type sequence window around each variant from the genome, applies the alternate allele, and compares the ensemble's predictions to derive delta scores for every overlapping gene. Here we use a small bundled synthetic genome (chromosome `1`, 60 kb) and a one-gene annotation so the notebook runs anywhere; real analyses use the full human genome (`grch37`/`grch38`) and the bundled GENCODE annotation.

In [4]:
config = SpliceAIScoreConfig(
    reference_fasta="example_genome.fa",
    annotation="example_annotation.txt",
    device="cpu",  # CPU so this runs on hosts without a GPU
)
inputs = SpliceAIScoreInput(
    variants=[SpliceAIVariant(chromosome="1", position=30000, ref="A", alt="C")]
)

result = run_spliceai_score(inputs, config)

variant_result = result.results[0]
print(f"Variant: chr{variant_result.chromosome}:{variant_result.position} {variant_result.ref}>{variant_result.alt}")
print(f"Max delta score: {variant_result.metrics['max_delta_score']}")
print("\nPer-gene delta scores:")
for gene in variant_result.scores:
    print(
        f"  {gene.symbol} (allele {gene.allele}): "
        f"AG={gene.ds_ag:.3f} AL={gene.ds_al:.3f} "
        f"DG={gene.ds_dg:.3f} DL={gene.ds_dl:.3f}"
    )

Running run_spliceai_score [00:00]

Variant: chr1:30000 A>C
Max delta score: 0.0

Per-gene delta scores:
  SPLICEAI_DEMO (allele C): AG=0.000 AL=0.000 DG=0.000 DL=0.000


## Raw splice-site prediction

Raw prediction runs directly on a DNA sequence and needs no reference genome. SpliceAI pads 5000 bp of context on each side internally, so the output covers every position of the input sequence with a `[neither, acceptor, donor]` probability triple.

In [5]:
import numpy as np

sequence = "ACGT" * 250  # 1000 bp
predict_result = run_spliceai_predict(
    SpliceAIPredictInput(sequences=[sequence]),
    SpliceAIPredictConfig(device="cpu"),
)

probs = np.array(predict_result.predictions[0])  # channels: [neither, acceptor, donor]
print(f"Prediction shape: {probs.shape}  # [len, 3] -> [neither, acceptor, donor]")
print(f"Sequence length: {len(sequence)}")
print(f"Max acceptor probability (channel 1): {probs[:, 1].max():.4f} at position {probs[:, 1].argmax()}")
print(f"Max donor probability (channel 2): {probs[:, 2].max():.4f} at position {probs[:, 2].argmax()}")

Running run_spliceai_predict [00:00]

Prediction shape: (1000, 3)  # [len, 3] -> [neither, acceptor, donor]
Sequence length: 1000
Max acceptor probability (channel 1): 0.0218 at position 730
Max donor probability (channel 2): 0.1007 at position 593


In [6]:
# Export the variant scores. Supported formats are listed by the output object.
import tempfile
from pathlib import Path

print(f"Supported formats: {result.output_format_options} (default: {result.output_format_default})")

with tempfile.TemporaryDirectory() as tmp_dir:
    result.export("spliceai_scores", export_path=tmp_dir, file_format="vcf")
    exported = Path(tmp_dir) / "spliceai_scores.vcf"
    print(f"Exported to {exported}")
    print(exported.read_text())

Supported formats: ['json', 'csv', 'vcf'] (default: json)
Exported to /var/folders/rs/6dqw0_k1125fl7f7_9h85hgh0000gn/T/tmps4nmtkpq/spliceai_scores.vcf
##fileformat=VCFv4.2
##INFO=<ID=SpliceAI,Number=.,Type=String,Description="SpliceAI variant annotation. These include delta scores (DS) and delta positions (DP) for acceptor gain (AG), acceptor loss (AL), donor gain (DG), and donor loss (DL). Format: ALLELE|SYMBOL|DS_AG|DS_AL|DS_DG|DS_DL|DP_AG|DP_AL|DP_DG|DP_DL">
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO
1	30000	.	A	C	.	.	SpliceAI=C|SPLICEAI_DEMO|0.00|0.00|0.00|0.00|-46|-24|-46|-24

